# NHANES 02 - Análisis exploratorio y factores asociados

Corresponde a la etapa de **Comprensión de los datos** del proceso CRISP-DM y al **objetivo específico 1**: identificar los factores asociados a HbA1c ≥ 5.7 % en adultos de 20 a 69 años sin diagnóstico previo de diabetes ni prediabetes.

Sigue las secciones 9 y 12 de `docs/protocolo_nhanes.md` (etiqueta `protocolo-nhanes-v1.1`):

- Descriptivos ponderados con el peso combinado; pruebas bivariadas sin ponderar.
- OR crudos y ajustados por regresión logística multivariable con todos los predictores (casos completos); OR por 10 años, 5 kg/m² y 10 cm.
- **Criterio de HE1:** edad, IMC, perímetro abdominal, hipertensión diagnosticada y antecedente familiar de diabetes se consideran asociados si su OR ajustado tiene p < 0.05; se reporta la dirección.
- VIF y modelos con IMC y perímetro abdominal por separado.

**Solo se usa el conjunto de desarrollo** (`data/processed/nhanes_desarrollo.csv`). El conjunto de prueba no se abre.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy import stats
from statsmodels.stats.outliers_influence import variance_inflation_factor

import preparacion_nhanes as pn
from preparacion_nhanes import DESENLACE, DESENLACE_SECUNDARIO, PREDICTORES, cargar_desarrollo

SALIDA = Path('../../resultados/nhanes/02_factores_asociados')
SALIDA.mkdir(parents=True, exist_ok=True)
pd.set_option('display.float_format', '{:.3f}'.format)

# etiqueta, tipo y unidades por las que se expresa el OR
VARIABLES = {
    'edad': ('Edad (años)', 'continua', 10),
    'hombre': ('Sexo masculino', 'binaria', 1),
    'imc': ('IMC (kg/m²)', 'continua', 5),
    'cintura': ('Perímetro abdominal (cm)', 'continua', 10),
    'hta': ('Hipertensión diagnosticada', 'binaria', 1),
    'familiar_dm': ('Antecedente familiar de diabetes', 'binaria', 1),
    'actividad': ('Actividad física recreativa', 'binaria', 1),
    'fuma': ('Fuma actualmente', 'binaria', 1),
}
FACTORES_HE1 = ['edad', 'imc', 'cintura', 'hta', 'familiar_dm']

d = cargar_desarrollo()
assert set(d['SEQN']).isdisjoint(pd.read_csv(pn.RUTA_PARTICION).query("conjunto == 'prueba'")['SEQN'])
print(f'Desarrollo: {len(d):,} participantes, {int(d[DESENLACE].sum()):,} con HbA1c >= 5.7 %, '
      f'{d.conglomerado.nunique()} conglomerados')

Desarrollo: 18,264 participantes, 4,899 con HbA1c >= 5.7 %, 163 conglomerados


La comprobación anterior solo lee la columna de asignación de `particion/particion_nhanes.csv` (que no contiene datos de salud) para confirmar que ningún participante de prueba está en el desarrollo.

## Prevalencia de HbA1c elevada

La prevalencia ponderada usa el peso combinado (`WTMEC2YR × 2 / 13.2`; `WTMECPRP × 3.2 / 13.2`); la cruda corresponde a la muestra sin ponderar.

In [2]:
def media_ponderada(x, w):
    ok = x.notna() & w.notna() & (w > 0)
    return float(np.average(x[ok], weights=w[ok]))


def fila_prevalencia(nombre, g):
    return {
        'grupo': nombre,
        'participantes': len(g),
        'HbA1c >= 5.7 cruda %': 100 * g[DESENLACE].mean(),
        'HbA1c >= 5.7 ponderada %': 100 * media_ponderada(g[DESENLACE], g['peso']),
        'HbA1c >= 6.5 cruda %': 100 * g[DESENLACE_SECUNDARIO].mean(),
        'HbA1c >= 6.5 ponderada %': 100 * media_ponderada(g[DESENLACE_SECUNDARIO], g['peso']),
    }


grupos_edad = pd.cut(d['edad'], [20, 40, 60, 70], right=False, labels=['20-39', '40-59', '60-69'])
prevalencia = pd.DataFrame(
    [fila_prevalencia('Total', d)]
    + [fila_prevalencia(f'Edad {e}', g) for e, g in d.groupby(grupos_edad, observed=True)]
    + [fila_prevalencia('Hombres' if s == 1 else 'Mujeres', g) for s, g in d.groupby('hombre')]
    + [fila_prevalencia(f'Ciclo {pn.CICLOS[c][2]}', g) for c, g in d.groupby('ciclo')]
)
prevalencia.to_csv(SALIDA / 'prevalencia.csv', index=False)
prevalencia

,grupo,participantes,HbA1c >= 5.7 cruda %,HbA1c >= 5.7 ponderada %,HbA1c >= 6.5 cruda %,HbA1c >= 6.5 ponderada %
0,Total,18264,26.823,20.615,2.360,1.495
1,Edad 20-39,8285,13.627,10.430,1.014,0.671
2,Edad 40-59,7006,33.129,25.546,3.126,2.120
3,Edad 60-69,2973,48.739,41.144,4.305,2.456
4,Mujeres,9304,25.451,20.293,2.107,1.261
5,Hombres,8960,28.248,20.942,2.623,1.732
6,Ciclo 2007-2008,2668,24.850,19.638,2.811,1.960
7,Ciclo 2009-2010,2941,29.548,24.221,2.482,1.525
8,Ciclo 2011-2012,2648,27.417,22.417,2.870,1.916
9,Ciclo 2013-2014,2731,23.984,18.398,1.611,1.035


## Características según el desenlace

Porcentajes y medias ponderados. Valor p de pruebas sin ponderar: chi-cuadrado para variables binarias y Mann-Whitney para continuas.

In [3]:
grupos = {'Total': d, 'HbA1c >= 5.7': d[d[DESENLACE] == 1], 'HbA1c < 5.7': d[d[DESENLACE] == 0]}
filas = []
for var, (etiqueta, tipo, _) in VARIABLES.items():
    fila = {'variable': etiqueta, 'medida': '% ponderado' if tipo == 'binaria' else 'media ponderada'}
    for nombre, g in grupos.items():
        valor = media_ponderada(g[var], g['peso'])
        fila[nombre] = 100 * valor if tipo == 'binaria' else valor
    if tipo == 'binaria':
        fila['p'] = stats.chi2_contingency(pd.crosstab(d[var], d[DESENLACE]))[1]
    else:
        fila['p'] = stats.mannwhitneyu(grupos['HbA1c >= 5.7'][var].dropna(), grupos['HbA1c < 5.7'][var].dropna()).pvalue
    filas.append(fila)

caracteristicas = pd.DataFrame(filas)
caracteristicas.to_csv(SALIDA / 'caracteristicas.csv', index=False)
caracteristicas

,variable,medida,Total,HbA1c >= 5.7,HbA1c < 5.7,p
0,Edad (años),media ponderada,41.906,49.730,39.874,0.000
1,Sexo masculino,% ponderado,49.575,50.362,49.371,0.000
2,IMC (kg/m²),media ponderada,28.549,31.255,27.846,0.000
3,Perímetro abdominal (cm),media ponderada,97.247,104.609,95.350,0.000
4,Hipertensión diagnosticada,% ponderado,22.010,34.209,18.843,0.000
5,Antecedente familiar de diabetes,% ponderado,35.788,44.195,33.595,0.000
6,Actividad física recreativa,% ponderado,58.352,46.566,61.413,0.000
7,Fuma actualmente,% ponderado,21.536,26.151,20.337,0.000


### Raza y etnia (solo descriptivo)

`RIDRETH1` no es predictor (protocolo, sección 4). Se describe porque la prevalencia de HbA1c elevada en NHANES difiere entre grupos, lo que se considera al discutir la aplicación en Lima.

In [4]:
RAZA = {1: 'Mexicano-estadounidense', 2: 'Otro hispano', 3: 'Blanco no hispano', 4: 'Negro no hispano', 5: 'Otra o multirracial'}
raza = pd.DataFrame([
    {'grupo': RAZA[r], 'participantes': len(g), '% de la muestra': 100 * len(g) / len(d),
     'HbA1c >= 5.7 ponderada %': 100 * media_ponderada(g[DESENLACE], g['peso'])}
    for r, g in d.groupby('raza_etnia')
])
raza.to_csv(SALIDA / 'raza_etnia.csv', index=False)
raza

,grupo,participantes,% de la muestra,HbA1c >= 5.7 ponderada %
0,Mexicano-estadounidense,2843,15.566,24.646
1,Otro hispano,1972,10.797,21.542
2,Blanco no hispano,6897,37.763,16.931
3,Negro no hispano,4117,22.542,35.216
4,Otra o multirracial,2435,13.332,24.803


## Datos faltantes

In [5]:
faltantes = pd.DataFrame({
    'variable': [VARIABLES[v][0] for v in VARIABLES],
    'faltantes': [int(d[v].isna().sum()) for v in VARIABLES],
    '%': [100 * d[v].isna().mean() for v in VARIABLES],
})
faltantes.to_csv(SALIDA / 'faltantes.csv', index=False)
faltantes[faltantes['faltantes'] > 0]

,variable,faltantes,%
2,IMC (kg/m²),149,0.816
3,Perímetro abdominal (cm),623,3.411
4,Hipertensión diagnosticada,13,0.071
5,Antecedente familiar de diabetes,283,1.549
7,Fuma actualmente,11,0.060


## Odds ratios crudos y ajustados

Regresión logística. Los OR ajustados provienen de un único modelo multivariable con todos los predictores, en casos completos. Las variables continuas se expresan por unidades interpretables (edad por cada 10 años, IMC por cada 5 kg/m², perímetro abdominal por cada 10 cm).

In [6]:
def escalar(datos, variables):
    return pd.DataFrame({v: datos[v] / VARIABLES[v][2] for v in variables}, index=datos.index)


def logit(datos, variables, **ajuste):
    X = sm.add_constant(escalar(datos, variables))
    return sm.Logit(datos[DESENLACE], X).fit(disp=False, **ajuste)


def tabla_or(modelo, variables):
    ic = modelo.conf_int()
    return pd.DataFrame({
        'OR': np.exp(modelo.params[variables]),
        'IC95_inf': np.exp(ic.loc[variables, 0]),
        'IC95_sup': np.exp(ic.loc[variables, 1]),
        'p': modelo.pvalues[variables],
    })


def etiqueta_or(var):
    etiqueta, tipo, escala = VARIABLES[var]
    return f'{etiqueta}, por cada {escala}' if tipo == 'continua' and escala != 1 else etiqueta


completos = d.dropna(subset=PREDICTORES)
print(f'Casos completos para el modelo multivariable: {len(completos):,} de {len(d):,}')

crudos = pd.concat([tabla_or(logit(d.dropna(subset=[v]), [v]), [v]) for v in PREDICTORES])
modelo_ajustado = logit(completos, PREDICTORES)
ajustados = tabla_or(modelo_ajustado, PREDICTORES)
odds_ratios = crudos.add_suffix('_crudo').join(ajustados.add_suffix('_ajustado'))
odds_ratios.insert(0, 'variable', [etiqueta_or(v) for v in odds_ratios.index])
odds_ratios.to_csv(SALIDA / 'odds_ratios.csv', index=False)
odds_ratios

Casos completos para el modelo multivariable: 17,324 de 18,264


,variable,OR_crudo,IC95_inf_crudo,IC95_sup_crudo,p_crudo,OR_ajustado,IC95_inf_ajustado,IC95_sup_ajustado,p_ajustado
edad,"Edad (años), por cada 10",1.747,1.702,1.794,0.000,1.780,1.726,1.837,0.000
hombre,Sexo masculino,1.153,1.080,1.231,0.000,1.328,1.222,1.442,0.000
imc,"IMC (kg/m²), por cada 5",1.407,1.373,1.441,0.000,1.490,1.383,1.604,0.000
cintura,"Perímetro abdominal (cm), por cada 10",1.404,1.374,1.434,0.000,0.981,0.921,1.045,0.552
hta,Hipertensión diagnosticada,2.288,2.128,2.459,0.000,1.048,0.962,1.142,0.279
familiar_dm,Antecedente familiar de diabetes,1.450,1.356,1.551,0.000,1.259,1.168,1.357,0.000
actividad,Actividad física recreativa,0.601,0.563,0.642,0.000,0.799,0.741,0.861,0.000
fuma,Fuma actualmente,1.162,1.077,1.254,0.000,1.337,1.225,1.459,0.000


### Errores estándar robustos por conglomerado (sensibilidad)

Los participantes de un mismo conglomerado no son independientes, y los errores estándar del modelo anterior suponen que sí lo son. Como análisis de sensibilidad se reestima el mismo modelo con errores estándar robustos por conglomerado. Los OR son idénticos; solo cambian los intervalos y los valores p. **El criterio de HE1 sigue siendo el del modelo principal** (protocolo, sección 9). Este análisis adicional se registró en la sección 15 del protocolo antes de ejecutarse.

In [7]:
modelo_robusto = logit(completos, PREDICTORES, cov_type='cluster',
                       cov_kwds={'groups': pd.factorize(completos['conglomerado'])[0]})
robustos = tabla_or(modelo_robusto, PREDICTORES)
robustos.insert(0, 'variable', [etiqueta_or(v) for v in robustos.index])
robustos.to_csv(SALIDA / 'odds_ratios_robustos_conglomerado.csv', index=False)
robustos

,variable,OR,IC95_inf,IC95_sup,p
edad,"Edad (años), por cada 10",1.780,1.724,1.839,0.000
hombre,Sexo masculino,1.328,1.229,1.434,0.000
imc,"IMC (kg/m²), por cada 5",1.490,1.379,1.609,0.000
cintura,"Perímetro abdominal (cm), por cada 10",0.981,0.920,1.047,0.561
hta,Hipertensión diagnosticada,1.048,0.969,1.134,0.241
familiar_dm,Antecedente familiar de diabetes,1.259,1.160,1.367,0.000
actividad,Actividad física recreativa,0.799,0.746,0.855,0.000
fuma,Fuma actualmente,1.337,1.218,1.467,0.000


## Colinealidad entre IMC y perímetro abdominal

Se reportan la correlación y los factores de inflación de la varianza (VIF). Como análisis de sensibilidad, se reestima el modelo ajustado excluyendo, por separado, cada una de las dos variables. Según el protocolo, este análisis **no cambia el criterio de HE1**.

In [8]:
print(f"Correlación IMC - perímetro abdominal: {completos['imc'].corr(completos['cintura']):.2f}")

Xv = sm.add_constant(completos[PREDICTORES])
vif = pd.DataFrame({
    'variable': [VARIABLES[v][0] for v in PREDICTORES],
    'VIF': [variance_inflation_factor(Xv.values, i + 1) for i in range(len(PREDICTORES))],
})
vif.to_csv(SALIDA / 'vif.csv', index=False)
vif

Correlación IMC - perímetro abdominal: 0.91


,variable,VIF
0,Edad (años),1.266
1,Sexo masculino,1.207
2,IMC (kg/m²),7.841
3,Perímetro abdominal (cm),8.139
4,Hipertensión diagnosticada,1.188
5,Antecedente familiar de diabetes,1.028
6,Actividad física recreativa,1.057
7,Fuma actualmente,1.047


In [9]:
sensibilidad = []
for excluida in ['cintura', 'imc']:
    variables = [v for v in PREDICTORES if v != excluida]
    t = tabla_or(logit(completos, variables), [v for v in FACTORES_HE1 if v != excluida])
    t.insert(0, 'modelo', f'sin {VARIABLES[excluida][0]}')
    sensibilidad.append(t)
sensibilidad = pd.concat(sensibilidad)
sensibilidad.insert(1, 'variable', [etiqueta_or(v) for v in sensibilidad.index])
sensibilidad.to_csv(SALIDA / 'sensibilidad_imc_cintura.csv', index=False)
sensibilidad

,modelo,variable,OR,IC95_inf,IC95_sup,p
edad,sin Perímetro abdominal (cm),"Edad (años), por cada 10",1.776,1.723,1.830,0.000
imc,sin Perímetro abdominal (cm),"IMC (kg/m²), por cada 5",1.459,1.418,1.501,0.000
hta,sin Perímetro abdominal (cm),Hipertensión diagnosticada,1.047,0.961,1.140,0.294
familiar_dm,sin Perímetro abdominal (cm),Antecedente familiar de diabetes,1.258,1.167,1.357,0.000
edad,sin IMC (kg/m²),"Edad (años), por cada 10",1.692,1.643,1.743,0.000
cintura,sin IMC (kg/m²),"Perímetro abdominal (cm), por cada 10",1.346,1.314,1.379,0.000
hta,sin IMC (kg/m²),Hipertensión diagnosticada,1.058,0.972,1.152,0.195
familiar_dm,sin IMC (kg/m²),Antecedente familiar de diabetes,1.271,1.179,1.370,0.000


## Contrastación de HE1

In [10]:
he1 = ajustados.loc[FACTORES_HE1].copy()
he1.insert(0, 'variable', [etiqueta_or(v) for v in FACTORES_HE1])
he1['asociado (p < 0.05)'] = np.where(he1['p'] < 0.05, 'Sí', 'No')
he1['dirección'] = np.where(he1['p'] >= 0.05, 'sin asociación significativa', np.where(he1['OR'] > 1, 'directa', 'inversa'))
he1['p robusto por conglomerado'] = robustos.loc[FACTORES_HE1, 'p']
he1.to_csv(SALIDA / 'he1.csv', index=False)
he1

,variable,OR,IC95_inf,IC95_sup,p,asociado (p < 0.05),dirección,p robusto por conglomerado
edad,"Edad (años), por cada 10",1.780,1.726,1.837,0.000,Sí,directa,0.000
imc,"IMC (kg/m²), por cada 5",1.490,1.383,1.604,0.000,Sí,directa,0.000
cintura,"Perímetro abdominal (cm), por cada 10",0.981,0.921,1.045,0.552,No,sin asociación significativa,0.561
hta,Hipertensión diagnosticada,1.048,0.962,1.142,0.279,No,sin asociación significativa,0.241
familiar_dm,Antecedente familiar de diabetes,1.259,1.168,1.357,0.000,Sí,directa,0.000


## Interpretación

**HE1 se cumple parcialmente.** Con el criterio prefijado (p < 0.05 en el modelo ajustado), tres de los cinco factores están asociados a HbA1c ≥ 5.7 %, todos con asociación directa:

- **Edad:** OR 1.78 por cada 10 años (IC 95 % 1.73–1.84).
- **IMC:** OR 1.49 por cada 5 kg/m² (1.38–1.60).
- **Antecedente familiar de diabetes:** OR 1.26 (1.17–1.36).

Los otros dos factores **no cumplen el criterio**:

- **Perímetro abdominal:** OR ajustado 0.98 por cada 10 cm, p = 0.55. IMC y perímetro abdominal están muy correlacionados (r = 0.91; VIF ≈ 8). Por eso, en el modelo con ambos, el OR del perímetro se interpreta *a igual IMC*: con un IMC dado, más centímetros de cintura no añaden asociación detectable. Al excluir el IMC, el perímetro abdominal sí se asocia (OR 1.35 por cada 10 cm; p < 0.001). En la práctica, ambas variables captan sobre todo la misma información, la adiposidad.
- **Hipertensión diagnosticada:** el OR crudo es 2.29, pero el ajustado es 1.05 (p = 0.28). Esa reducción indica que la asociación cruda coincide en gran parte con la de los demás predictores del modelo, principalmente la edad y la adiposidad. El diagnóstico de hipertensión no añade información independiente sobre la HbA1c una vez que se conocen esas variables. El análisis de sensibilidad (excluyendo el IMC o el perímetro abdominal) no cambia esta conclusión.

**Otros predictores** (no forman parte de HE1, pero entran en el modelo): el sexo masculino (OR 1.33) y el tabaquismo actual (OR 1.34) se asocian de forma directa, y la actividad física recreativa de forma inversa (OR 0.80). Son asociaciones estadísticas en datos transversales: **no indican efectos causales**. En particular, NHANES no permite saber si la actividad física es anterior o posterior a la alteración de la HbA1c.

**Errores estándar robustos por conglomerado** (sensibilidad): los OR son los mismos y los valores p apenas cambian, sin alterar ninguna conclusión de HE1.

**Prevalencia.** La prevalencia cruda (26.8 %) es mayor que la ponderada (20.6 %) porque NHANES sobrerrepresenta deliberadamente a ciertos grupos, como las personas mayores y las poblaciones negra e hispana. La ponderada estima la de la población de EE. UU. de 20 a 69 años sin diagnóstico previo. La prevalencia varía según la raza y la etnia (por ejemplo, 16.9 % en blancos no hispanos y 35.2 % en negros no hispanos), aunque esta variable no es predictor. Ese hecho se retoma en la discusión sobre la aplicación del modelo en Lima.

Estos resultados se obtuvieron solo con el conjunto de desarrollo; el conjunto de prueba no se utilizó.
